In [1]:
# using the ibm dataset, we are creating new columns.
import pandas as pd
import numpy as np

ibm = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Resample 1000 rows, preserving distribution shape
df = ibm.sample(n=1000, replace=False, random_state=42).reset_index(drop=True)

# --- Remap IBM columns to blueprint state variables ---

# satisfaction_score: average of 3 satisfaction dimensions, rescale to 1-10
df['satisfaction_score'] = (
    df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction']
) / 3 * 2.5   # converts 1-4 range to 1-10 by multiplying by 2.5

# productivity_baseline: performance × involvement, normalised to 0-1
df['productivity_baseline'] = (
    (df['PerformanceRating'] / 4) * 0.5 +
    (df['JobInvolvement']    / 4) * 0.5
)

# resistance_propensity: low satisfaction + high tenure = high resistance
df['resistance_propensity'] = (
    (1 - df['JobSatisfaction'] / 4) * 0.5 +
    np.log1p(df['YearsAtCompany']) / np.log1p(40) * 0.5
).clip(0, 1)

# training_history: direct rescale
df['training_times_yr'] = df['TrainingTimesLastYear']

# attrition risk flag (for churn modelling in ABM)
# attrition: the natural reduction of employees over time
df['churn_risk_flag'] = (df['Attrition'] == 'Yes').astype(int)

# Keep only blueprint-relevant columns
df = df[['satisfaction_score', 'productivity_baseline', 'resistance_propensity',
         'training_times_yr', 'churn_risk_flag', 'Education', 'Age',
         'YearsAtCompany', 'WorkLifeBalance']]


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Features to cluster on — all normalised
features = ['satisfaction_score', 'productivity_baseline',
            'resistance_propensity', 'training_times_yr']

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# K=5 — maps to Rogers 5-persona framework
kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Inspect centroids to assign persona labels
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                         columns=features)
print(centroids.sort_values('resistance_propensity'))

# Map cluster IDs to persona names based on centroid inspection
# (your mapping will differ — inspect your centroids first)
PERSONA_MAP = {
    0: 'Tech Pioneer',
    1: 'Power User',
    2: 'Pragmatic Adopter',
    3: 'Reluctant User',
    4: 'Remote-First Worker'
}
df['persona'] = df['cluster'].map(PERSONA_MAP)

# Derive digital_dexterity from Education + training + inverse resistance
df['digital_dexterity'] = (
    df['Education']       / 5 * 3   +   # 0-3 pts
    df['training_times_yr'] / 6 * 3 +   # 0-3 pts
    (1 - df['resistance_propensity']) * 4  # 0-4 pts
).clip(1, 10)


   satisfaction_score  productivity_baseline  resistance_propensity  \
1            7.902299               0.592672               0.277009   
3            7.925425               0.783784               0.300443   
0            6.782561               0.748344               0.385318   
4            5.730874               0.607582               0.531943   
2            5.509592               0.791367               0.546913   

   training_times_yr  
1           2.534483  
3           2.264264  
0           5.099338  
4           2.500000  
2           2.460432  


In [4]:
# M365-equivalent collaboration signals — stratified by persona
# Benchmark: Microsoft 365 Productivity Report — avg 40 emails/day knowledge worker

COLLAB_PARAMS = {
    'Tech Pioneer':        {'email': (52, 9),  'meetings': (11, 2.5), 'teams_msg': (38, 8)},
    'Power User':          {'email': (61, 11), 'meetings': (13, 3.0), 'teams_msg': (45, 10)},
    'Pragmatic Adopter':   {'email': (36, 8),  'meetings': (8,  2.0), 'teams_msg': (22, 6)},
    'Reluctant User':      {'email': (19, 6),  'meetings': (5,  1.8), 'teams_msg': (11, 4)},
    'Remote-First Worker': {'email': (31, 7),  'meetings': (9,  2.2), 'teams_msg': (34, 9)},
}

for persona, params in COLLAB_PARAMS.items():
    mask = df['persona'] == persona
    n    = mask.sum()
    df.loc[mask, 'email_vol_daily']    = np.random.normal(*params['email'],    n).clip(2, 120)
    df.loc[mask, 'meetings_per_week']  = np.random.normal(*params['meetings'], n).clip(0, 25)
    df.loc[mask, 'teams_msg_daily']    = np.random.normal(*params['teams_msg'],n).clip(0, 100)

# Derive collaboration_density as composite (0-1)
df['collab_density'] = (
    df['email_vol_daily']   / 120 * 0.3 +
    df['meetings_per_week'] / 25  * 0.4 +
    df['teams_msg_daily']   / 100 * 0.3
).clip(0, 1)


In [5]:
# ITSM ticket signals — Poisson distributed (realistic arrival process)
# Reluctant Users generate 2.2x the org average — calibrated to simulation walkthrough

TICKET_PARAMS = {          # (lambda for Poisson)
    'Tech Pioneer':        0.8,    # self-sufficient — rarely raises tickets
    'Power User':          1.1,    # uses advanced features, occasional edge cases
    'Pragmatic Adopter':   2.1,    # moderate support need
    'Reluctant User':      5.2,    # HIGH — this is the blast radius in chatbot scenario
    'Remote-First Worker': 2.6,    # elevated due to remote infrastructure issues
}

CATEGORIES = ['Password Reset', 'Software Install', 'VPN / Network',
              'App Crash', 'Hardware', 'Access / Permissions', 'Training Request']

# Category probabilities differ by persona (Reluctants have more App Crash / Hardware)
CAT_PROBS = {
    'Tech Pioneer':        [0.10, 0.20, 0.15, 0.10, 0.10, 0.25, 0.10],
    'Power User':          [0.08, 0.25, 0.12, 0.15, 0.08, 0.22, 0.10],
    'Pragmatic Adopter':   [0.20, 0.18, 0.18, 0.14, 0.12, 0.12, 0.06],
    'Reluctant User':      [0.28, 0.12, 0.16, 0.22, 0.14, 0.06, 0.02],
    'Remote-First Worker': [0.15, 0.14, 0.28, 0.12, 0.18, 0.10, 0.03],
}

for persona, lam in TICKET_PARAMS.items():
    mask = df['persona'] == persona
    n    = mask.sum()
    df.loc[mask, 'tickets_per_month'] = np.random.poisson(lam, n)
    df.loc[mask, 'resolution_hrs']    = np.random.exponential(scale=6, size=n).clip(0.5, 48)
    df.loc[mask, 'ticket_category']   = np.random.choice(
        CATEGORIES, size=n, p=CAT_PROBS[persona])
    df.loc[mask, 'reopened_rate']     = np.random.beta(1.2, 8, n)  # ~13% reopen avg

# Derive support_dependency (0-1)
df['support_dependency'] = (df['tickets_per_month'] / 8).clip(0, 1)

# Derive frustration_level from resolution time + reopen rate
df['frustration_level'] = (
    df['resolution_hrs'] / 48 * 0.6 +
    df['reopened_rate']        * 0.4
).clip(0, 1)


In [6]:
# DEX signals — Beta distributed crash rates (right-skewed, mostly low)
# Reluctant Users experience more crashes due to lower dexterity + older hardware

DEX_PARAMS = {
    #                     crash_rate(a,b)  load_time_sec(mean,std)  session_min(mean,std)
    'Tech Pioneer':        ((1.0, 18),      (1.8, 0.6),              (55, 14)),
    'Power User':          ((1.1, 16),      (2.1, 0.7),              (62, 16)),
    'Pragmatic Adopter':   ((1.5, 12),      (3.2, 1.1),              (42, 18)),
    'Reluctant User':      ((2.2,  8),      (5.1, 1.8),              (28, 12)),
    'Remote-First Worker': ((1.6, 10),      (4.0, 1.6),              (38, 16)),
}

for persona, (cr, lt, sd) in DEX_PARAMS.items():
    mask = df['persona'] == persona
    n    = mask.sum()
    df.loc[mask, 'app_crash_rate']      = np.random.beta(*cr, n)
    df.loc[mask, 'avg_load_time_sec']   = np.random.normal(*lt, n).clip(0.5, 15)
    df.loc[mask, 'session_duration_min']= np.random.normal(*sd, n).clip(5, 120)

# Contribute to friction_level (adds to ITSM-derived frustration)
df['friction_level'] = (
    df['frustration_level']  * 0.5 +
    df['app_crash_rate']     * 0.3 +
    (df['avg_load_time_sec'] / 15) * 0.2
).clip(0, 1)


In [7]:
# Survey / NPS signals — normally distributed per persona
# eNPS scale: -100 to +100. Org average target: ~+22 (Qualtrics benchmark)

SURVEY_PARAMS = {
    #                     eNPS(mean,std)   dex_feedback(1-10)  pulse_sat(1-5)
    'Tech Pioneer':        ((72, 14),        (8.4, 0.8),         (4.2, 0.6)),
    'Power User':          ((58, 16),        (7.6, 1.0),         (3.9, 0.7)),
    'Pragmatic Adopter':   ((28, 22),        (5.8, 1.4),         (3.2, 0.9)),
    'Reluctant User':      ((-18, 28),       (3.1, 1.6),         (2.2, 1.0)),
    'Remote-First Worker': ((38, 20),        (6.2, 1.3),         (3.5, 0.8)),
}

for persona, (enps, dex, pulse) in SURVEY_PARAMS.items():
    mask = df['persona'] == persona
    n    = mask.sum()
    df.loc[mask, 'enps_score']      = np.random.normal(*enps,  n).clip(-100, 100)
    df.loc[mask, 'dex_feedback']    = np.random.normal(*dex,   n).clip(1, 10)
    df.loc[mask, 'pulse_sat']       = np.random.normal(*pulse, n).clip(1, 5)

# Validate: org-level eNPS should be ~+22
org_enps = df['enps_score'].mean()
print(f'Org-level eNPS: {org_enps:.1f}  (target: ~+22)')
# Adjust persona means if org average drifts too far from benchmark


Org-level eNPS: 23.2  (target: ~+22)


In [8]:
# IAM / SSO signals — Poisson logins, Beta activation rates

SSO_PARAMS = {
    #                     logins/day(λ)  failed/week(λ)  activation_rate(a,b)
    'Tech Pioneer':        (6.2,          0.4,            (9.0, 1.0)),   # ~90% activation
    'Power User':          (5.8,          0.6,            (8.0, 1.5)),   # ~84% activation
    'Pragmatic Adopter':   (4.1,          1.1,            (5.5, 3.5)),   # ~61% activation
    'Reluctant User':      (2.8,          2.4,            (2.5, 4.5)),   # ~36% activation
    'Remote-First Worker': (4.8,          1.4,            (5.0, 3.0)),   # ~63% activation
}

for persona, (lpd, flw, act) in SSO_PARAMS.items():
    mask = df['persona'] == persona
    n    = mask.sum()
    df.loc[mask, 'logins_per_day']    = np.random.poisson(lpd,  n)
    df.loc[mask, 'failed_logins_wk']  = np.random.poisson(flw,  n)
    df.loc[mask, 'app_activation_rt'] = np.random.beta(*act, n)

# Validate: org-level activation should be ~61% (known benchmark gap = 39%)
print(f'Org activation rate: {df["app_activation_rt"].mean():.1%}  (target: ~61%)')


Org activation rate: 59.8%  (target: ~61%)


In [9]:
# LMS signals — normally distributed per persona

LMS_PARAMS = {
    #                     completion(mean,std)  score(mean,std)  time_hrs(mean,std)
    'Tech Pioneer':        ((0.91, 0.07),        (86, 9),          (3.1, 1.0)),
    'Power User':          ((0.84, 0.09),        (82, 10),         (3.8, 1.2)),
    'Pragmatic Adopter':   ((0.64, 0.14),        (72, 12),         (5.8, 1.8)),
    'Reluctant User':      ((0.38, 0.16),        (58, 15),         (8.2, 2.4)),
    'Remote-First Worker': ((0.71, 0.13),        (75, 11),         (5.2, 1.6)),
}

for persona, (comp, score, hrs) in LMS_PARAMS.items():
    mask = df['persona'] == persona
    n    = mask.sum()
    df.loc[mask, 'lms_completion']     = np.random.normal(*comp,  n).clip(0, 1)
    df.loc[mask, 'assessment_score']   = np.random.normal(*score, n).clip(0, 100)
    df.loc[mask, 'time_to_complete_hr']= np.random.normal(*hrs,   n).clip(0.5, 20)

# Final digital_dexterity composite — all Layer C signals now available
df['digital_dexterity'] = (
    df['digital_dexterity']    * 0.35 +   # base from Layer B
    df['app_activation_rt']    * 10 * 0.25 +  # SSO signal
    df['lms_completion']       * 10 * 0.20 +  # LMS signal
    (1 - df['app_crash_rate']) * 10 * 0.10 +  # DEX signal (inverse)
    df['dex_feedback']              * 0.10    # survey signal
).clip(1, 10)

# Validate: Tech Pioneer mean should be ~8.5, Reluctant User ~3.1
print(df.groupby('persona')['digital_dexterity'].mean().sort_values())


persona
Reluctant User         4.714230
Pragmatic Adopter      5.974644
Remote-First Worker    6.139610
Power User             7.540990
Tech Pioneer           8.162577
Name: digital_dexterity, dtype: float64


In [10]:
# Quick validation script

import matplotlib.pyplot as plt


print('=== BENCHMARK VALIDATION ===')

print(f'Org eNPS:           {df["enps_score"].mean():.1f}     (target: ~+22)')

print(f'Org ticket/month:   {df["tickets_per_month"].mean():.2f}  (target: ~2.4)')

print(f'Org app activation: {df["app_activation_rt"].mean():.1%}  (target: ~61%)')

print(f'LMS completion:     {df["lms_completion"].mean():.1%}    (target: ~65%)')

print(f'Avg assessment:     {df["assessment_score"].mean():.1f}   (target: ~74)')


print('\n=== PERSONA COUNTS ===')

print(df['persona'].value_counts())

# Target: Pioneer~50, PowerUser~150, Pragmatic~300, Reluctant~300, Remote~200


print('\n=== DIGITAL DEXTERITY BY PERSONA ===')

print(df.groupby('persona')['digital_dexterity'].agg(['mean','std']).round(2))

# Tech Pioneer mean: ~8.5 | Reluctant User mean: ~3.1


print('\n=== CORRELATION CHECK ===')

# digital_dexterity and tickets_per_month should be NEGATIVE (r ~ -0.55 to -0.70)

print(df[['digital_dexterity','tickets_per_month']].corr())

=== BENCHMARK VALIDATION ===
Org eNPS:           23.2     (target: ~+22)
Org ticket/month:   2.88  (target: ~2.4)
Org app activation: 59.8%  (target: ~61%)
LMS completion:     62.3%    (target: ~65%)
Avg assessment:     71.0   (target: ~74)

=== PERSONA COUNTS ===
persona
Reluctant User         334
Pragmatic Adopter      277
Tech Pioneer           151
Remote-First Worker    122
Power User             116
Name: count, dtype: int64

=== DIGITAL DEXTERITY BY PERSONA ===
                     mean   std
persona                        
Power User           7.54  0.47
Pragmatic Adopter    5.97  0.57
Reluctant User       4.71  0.63
Remote-First Worker  6.14  0.59
Tech Pioneer         8.16  0.47

=== CORRELATION CHECK ===
                   digital_dexterity  tickets_per_month
digital_dexterity           1.000000          -0.596643
tickets_per_month          -0.596643           1.000000
